# 🚀 Day 4: Guided Lab — Your First AI Agent with Function Calling

## Learning Objectives

By the end of this lab, you will be able to:
- Define Python functions as tools for the Gemini API
- Use function calling to let the model invoke your tools
- Control tool behavior with AUTO, ANY, and NONE modes
- Build a manual agent loop with conversation history
- Create a multi-tool agent combining CRM, calculator, and search
- Evaluate agent performance with test cases

---
## Part 0: Setup & Connection Test (10 min)

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os, json, time
from datetime import datetime, timezone
from google import genai
from google.genai import types

# ── API Key ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-2.5-flash-lite"

In [ ]:
# ── Logging Infrastructure ───────────────────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def log_interaction(role, content, label=None):
    """Log an agent interaction."""
    entry = {
        "ts": _now(),
        "role": role,
        "content": content if isinstance(content, str) else json.dumps(content),
        "label": label or "",
    }
    PROMPT_LOG.append(entry)
    return entry

def show_log(n=10):
    """Display the last n log entries."""
    import pandas as pd
    if not PROMPT_LOG:
        print("No interactions logged yet.")
        return
    df = pd.DataFrame(PROMPT_LOG[-n:])
    from IPython.display import display
    display(df)

In [ ]:
# ── Test Connection ──────────────────────────────────────
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Say 'Hello from Gemini!' in one sentence.",
)
print(f"✅ Connected to {MODEL_ID}")
print(f"   Response: {response.text}")

---
## Part 1: Your First Tool — Single Function Calling (10 min)

In function calling, you:
1. **Define a Python function** with type hints and a docstring
2. **Pass it as a tool** to `generate_content()`
3. **The model decides** whether and how to call it

> The model never executes your code — it only *requests* a call. Your code runs the function and returns the result.

In [ ]:
# ── Define a Tool ────────────────────────────────────────
def get_weather(location: str) -> dict:
    """Get the current weather for a location.

    Args:
        location: City name, e.g. 'New York', 'London', 'Tokyo'

    Returns:
        A dict with temperature, condition, and humidity.
    """
    weather_db = {
        "New York": {"temp_celsius": 22, "condition": "Sunny", "humidity": 65},
        "London":   {"temp_celsius": 13, "condition": "Rainy", "humidity": 85},
        "Tokyo":    {"temp_celsius": 20, "condition": "Cloudy", "humidity": 70},
    }
    result = weather_db.get(location)
    if result:
        return {**result, "location": location, "status": "ok"}
    return {"error": f"No weather data for '{location}'", "status": "not_found"}

# Test the function directly (no LLM yet)
print(get_weather("London"))
print(get_weather("Paris"))   # Not in our database

In [ ]:
# ── Call the Model with a Tool ───────────────────────────
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the weather like in London today?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
    ),
)

print("Response text:", response.text)

In [ ]:
# ── Inspect the Response Parts ───────────────────────────
# With automatic function calling (the default), the SDK runs
# the entire loop behind the scenes: it calls get_weather,
# feeds the result back, and returns ONLY the final text.
# That's why we see just 1 part — the intermediate function_call
# and function_response are consumed internally.

print("With automatic function calling (default):")
print(f"  Number of parts: {len(response.candidates[0].content.parts)}")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"  Part {i}: Text = {part.text[:200]}")

print("\n--- Now let's DISABLE automatic calling to see what really happens ---\n")

# Same request, but this time the SDK stops and hands us the function_call
raw_response = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the weather like in London today?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True  # SDK returns the function_call instead of executing it
        ),
    ),
)

print("With automatic function calling DISABLED:")
print(f"  Number of parts: {len(raw_response.candidates[0].content.parts)}")
for i, part in enumerate(raw_response.candidates[0].content.parts):
    print(f"\n  Part {i}:")
    if part.text:
        print(f"    Text: {part.text[:200]}")
    if part.function_call:
        print(f"    Function call: {part.function_call.name}")
        print(f"    Arguments: {dict(part.function_call.args)}")
        print("    → The model REQUESTED this call. It did NOT execute it.")
        print("      In Part 3, we'll build a loop that handles this ourselves.")

---
## Part 2: Controlling Tool Behavior (10 min)

Tool calling **modes** control how the model interacts with tools:

| Mode | Behavior | Use When |
|------|----------|----------|
| `AUTO` (default) | Model decides whether to call a tool or respond directly | General use |
| `ANY` | Model **must** call a tool | Force structured output or routing |
| `NONE` | Model cannot call any tools | Pure text response |

In [ ]:
# ── AUTO Mode: Model Decides ─────────────────────────────
print("=== AUTO Mode ===")
print("Query about a city IN our database:")
r1 = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the weather in Tokyo?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode="AUTO")
        ),
    ),
)
print(f"  → {r1.text}\n")

print("Query about something unrelated to weather:")
r2 = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the capital of France?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode="AUTO")
        ),
    ),
)
print(f"  → {r2.text}")

In [ ]:
# ── ANY Mode: Must Call a Tool ────────────────────────────
print("=== ANY Mode ===")
print("The model is FORCED to call get_weather, even for an unrelated question:\n")

r3 = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the capital of France?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode="ANY")
        ),
        # Disable auto-execution so we can see the raw function call
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)

for part in r3.candidates[0].content.parts:
    if part.function_call:
        print(f"  Tool called: {part.function_call.name}")
        print(f"  Arguments:   {dict(part.function_call.args)}")

In [ ]:
# ── NONE Mode: No Tools Allowed ──────────────────────────
print("=== NONE Mode ===")
print("Even for a weather question, the model must answer from its own knowledge:\n")

r4 = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the weather in London?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(mode="NONE")
        ),
    ),
)
print(f"  → {r4.text}")

---
## Part 3: Building the Agent Loop (15 min)

Automatic function calling is convenient, but for real agents you want **full control**. The manual agent loop:

1. Send user message to the model
2. If the model requests a tool call → execute it, send the result back
3. Repeat until the model gives a final text answer
4. Stop after `max_steps` as a safety limit

This is the **core pattern** for all agent architectures (ReAct, Plan-and-Execute, etc.).

In [ ]:
# ── The Agent Loop ───────────────────────────────────────
def run_agent(user_message, tools, system_prompt=None, max_steps=10):
    """A manual agent loop with full visibility.

    Args:
        user_message: The user's request.
        tools: List of Python functions to use as tools.
        system_prompt: Optional system instruction for the agent.
        max_steps: Maximum number of reasoning steps (safety limit).

    Returns:
        A tuple of (final_text, tools_called, trace) where:
        - final_text: The agent's final response string
        - tools_called: List of tool names invoked during the run
        - trace: List of dicts with 'call', 'tool', 'args', 'result' per tool call
    """
    tool_map = {fn.__name__: fn for fn in tools}
    call_count = 0        # Track total tool calls across all steps
    tools_called = []     # Record which tools were actually used
    trace = []            # Structured log: tool, args, result per call

    # Build initial contents
    contents = []
    if system_prompt:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=f"System: {system_prompt}\n\nUser: {user_message}")]
        ))
    else:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        ))

    log_interaction("user", user_message, label="agent_input")

    for step in range(max_steps):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=types.GenerateContentConfig(
                tools=tools,
                # Tool calling mode defaults to AUTO — the model
                # reasons about whether to use tools on each turn.
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True  # Model still reasons about tools —
                    # but the SDK won't execute them automatically.
                    # Instead it returns the function_call to us,
                    # and WE run the function below.
                ),
            ),
        )

        # Guard: the model may return an empty response
        # (e.g. rate limit, safety filter, transient error)
        parts = response.parts or []
        if not parts:
            print(f"  Step {step+1}: ⚠️ Empty response from model — retrying...")
            continue

        # Add model response to history
        contents.append(types.Content(role="model", parts=parts))

        # Check for function calls
        function_results = []
        for part in parts:
            if part.function_call:
                call_count += 1
                name = part.function_call.name
                args = dict(part.function_call.args)
                tools_called.append(name)
                print(f"  Tool call {call_count}: 🔧 {name}({args})")

                # Execute the function
                try:
                    result = tool_map[name](**args)
                except Exception as e:
                    result = {"error": str(e)}

                print(f"           → {result}")
                trace.append({
                    "call": call_count,
                    "tool": name,
                    "args": args,
                    "result": result if isinstance(result, str) else json.dumps(result),
                })
                log_interaction("tool", f"{name}({args}) → {result}", label="tool_call")

                function_results.append(
                    types.Part(
                        function_response=types.FunctionResponse(
                            name=name,
                            response={"result": result},
                        )
                    )
                )

        if function_results:
            contents.append(types.Content(role="user", parts=function_results))
        else:
            # No function calls → model is done
            final_text = response.text or "(no text response)"
            log_interaction("agent", final_text, label="agent_output")
            return final_text, tools_called, trace

    return "⚠️ Agent reached maximum steps without completing.", tools_called, trace

print("✅ run_agent() defined.")

In [ ]:
# ── Test: Simple Query ───────────────────────────────────
print("Query: 'What's the weather in London?'\n")
answer, tools_used, trace = run_agent(
    "What's the weather in London?",
    tools=[get_weather],
)
print(f"\n📝 Final Answer:\n{answer}")
print(f"🔧 Tools used: {tools_used}")

In [ ]:
# ── Test: Query Requiring Reasoning ──────────────────────
print("Query: 'Compare weather in New York and Tokyo. Where's better for a picnic?'\n")
answer, tools_used, trace = run_agent(
    "Compare the weather in New York and Tokyo. "
    "Which city would you recommend for an outdoor picnic today?",
    tools=[get_weather],
)
print(f"\n📝 Final Answer:\n{answer}")
print(f"🔧 Tools used: {tools_used}")

---
## Part 4: Multi-Tool Agent (15 min)

Real agents combine multiple tools. The model decides **which** tool to call (or whether to call one at all) based on the tool descriptions.

We'll add two more tools:
- **`get_customer_info`**: Look up customer account details (simulating a CRM)
- **`calculate`**: Evaluate mathematical expressions

In [ ]:
# ── Tool: Customer Lookup ────────────────────────────────
def get_customer_info(customer_id: str) -> dict:
    """Look up a customer's account details from the CRM.

    Args:
        customer_id: The unique customer ID, e.g. 'CUST-1234'

    Returns:
        A dict with name, plan, and monthly recurring revenue (MRR).
    """
    crm_database = {
        "CUST-1234": {"name": "Acme Corp",  "plan": "Enterprise", "mrr": 12000, "industry": "Manufacturing"},
        "CUST-5678": {"name": "TechStart",  "plan": "Growth",     "mrr": 3500,  "industry": "SaaS"},
        "CUST-9012": {"name": "RetailMax",  "plan": "Starter",    "mrr": 800,   "industry": "Retail"},
    }
    result = crm_database.get(customer_id)
    if result:
        return {**result, "customer_id": customer_id, "status": "found"}
    return {"error": f"Customer '{customer_id}' not found", "status": "not_found"}

print(get_customer_info("CUST-1234"))

In [ ]:
# ── Tool: Calculator ─────────────────────────────────────
import re

def calculate(expression: str) -> str:
    """Evaluate a mathematical expression using actual numbers and operators.

    Args:
        expression: A math expression with numeric values and operators only.
                    Do NOT use variable names — substitute the actual numbers.
                    Supports +, -, *, /, **, (). Example: '12000 * 12 * 0.85'

    Returns:
        The numeric result as a string, or an error message.
    """
    # Check for variable names (letters that aren't part of numbers)
    # Allow digits, operators, whitespace, parentheses, and decimal points
    if re.search(r'[a-zA-Z_]', expression):
        return (
            f"Error: expression contains variable names: '{expression}'. "
            f"Please substitute the actual numeric values and try again."
        )
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}. Use only numbers and operators (+, -, *, /, **, parentheses)."

print(calculate("12000 * 12 * 0.85"))
print(calculate("MRR * 12"))  # Shows the error the model would see

In [ ]:
# ── Multi-Tool Agent ─────────────────────────────────────
print("Query: 'What plan is CUST-1234 on? What would their annual cost be with 15% discount?'\n")

answer, tools_used, trace = run_agent(
    "What plan is customer CUST-1234 on? "
    "What would their annual cost be if we gave them a 15% discount?",
    tools=[get_customer_info, calculate],
)
print(f"\n📝 Final Answer:\n{answer}")
print(f"🔧 Tools used: {tools_used}")

In [ ]:
# ── Complex Multi-Tool Query ─────────────────────────────
print("Query: 'Compare MRR of CUST-1234 and CUST-5678. What's combined annual revenue?'\n")

answer, tools_used, trace = run_agent(
    "Compare the MRR of customers CUST-1234 and CUST-5678. "
    "Which one pays more? What's their combined annual revenue?",
    tools=[get_customer_info, calculate],
)
print(f"\n📝 Final Answer:\n{answer}")
print(f"🔧 Tools used: {tools_used}")

In [ ]:
# ── Agent with System Prompt ─────────────────────────────
print("Query: 'Tell me about CUST-9012 and check the weather in New York.'\n")
print("This time we add a system prompt to guide the agent's behavior.\n")

answer, tools_used, trace = run_agent(
    "Tell me about customer CUST-9012 and check the weather in their area (New York).",
    tools=[get_weather, get_customer_info, calculate],
    system_prompt=(
        "You are a helpful business assistant. "
        "Use tools to look up information. "
        "Always cite the source of your data (tool name). "
        "If a tool returns an error, tell the user clearly."
    ),
)
print(f"\n📝 Final Answer:\n{answer}")
print(f"🔧 Tools used: {tools_used}")

---
## Part 5: Agent Evaluation Basics (15 min)

How do you know your agent is working correctly? Build **test cases** with:
- An input query
- The tools you expect the agent to call
- Keywords the answer should contain
- A maximum step limit

In [ ]:
# ── Define Test Cases ────────────────────────────────────
# Note: Keywords are checked against the model's FINAL ANSWER (case-insensitive).
# The model rephrases tool results, so use words it's likely to say,
# not the exact strings from the tool output.

test_cases = [
    {
        "id": "T1",
        "input": "What's the weather in New York?",
        "expected_tools": ["get_weather"],
        "expected_keywords": ["sunny", "22"],
        "max_steps": 3,
        "description": "Simple single-tool query",
    },
    {
        "id": "T2",
        "input": "What's the weather in Paris?",
        "expected_tools": ["get_weather"],
        "expected_keywords": ["paris"],  # Model should mention Paris; it will rephrase the error
        "max_steps": 3,
        "description": "Query for missing data — should handle gracefully",
    },
    {
        "id": "T3",
        "input": "What plan is customer CUST-5678 on?",
        "expected_tools": ["get_customer_info"],
        "expected_keywords": ["growth"],
        "max_steps": 3,
        "description": "CRM lookup",
    },
    {
        "id": "T4",
        "input": "How much would CUST-1234 pay annually with a 20% discount?",
        "expected_tools": ["get_customer_info", "calculate"],
        "expected_keywords": ["115200", "115,200"],  # 12000 * 12 * 0.8 — model may format with comma
        "max_steps": 5,
        "description": "Multi-tool: lookup + calculation (may fail — discuss why!)",
    },
    {
        "id": "T5",
        "input": "What is the meaning of life?",
        "expected_tools": [],
        "expected_keywords": [],
        "max_steps": 2,
        "description": "No tools needed — pure text response",
    },
]

print(f"Defined {len(test_cases)} test cases.")

In [ ]:
# ── Run Test Cases ───────────────────────────────────────
all_tools = [get_weather, get_customer_info, calculate]
results = []

for test in test_cases:
    print(f"\n{'='*60}")
    print(f"Test {test['id']}: {test['description']}")
    print(f"Input: {test['input']}")

    answer, tools_called, trace = run_agent(
        test["input"],
        tools=all_tools,
        max_steps=test["max_steps"],
    )

    # Show the final answer
    print(f"\n  Answer: {(answer or '(none)')[:200]}")

    # Check tools used
    expected_tools = set(test["expected_tools"])
    actual_tools = set(tools_called)
    tools_match = expected_tools == actual_tools
    if not tools_match:
        print(f"  Tools expected: {sorted(expected_tools)}")
        print(f"  Tools actual:   {sorted(actual_tools)}")

    # Check keywords (at least one match is enough)
    answer_lower = (answer or "").lower()
    keywords_found = [kw for kw in test["expected_keywords"] if kw.lower() in answer_lower]
    keywords_missing = [kw for kw in test["expected_keywords"] if kw.lower() not in answer_lower]

    if test["expected_keywords"]:
        passed = len(keywords_found) >= 1  # At least one keyword match
    else:
        passed = True  # No keywords to check

    status = "✅ PASS" if passed else "❌ FAIL"

    results.append({
        "test_id": test["id"],
        "description": test["description"],
        "passed": passed,
        "tools_match": tools_match,
        "tools_called": sorted(actual_tools),
        "keywords_found": len(keywords_found),
        "keywords_total": len(test["expected_keywords"]),
        "answer_preview": (answer or "")[:100],
    })

    print(f"\n  {status} | Keywords: {len(keywords_found)}/{len(test['expected_keywords'])} | Tools match: {'✅' if tools_match else '❌'}")
    if keywords_missing:
        print(f"  Missing keywords: {keywords_missing}")

In [ ]:
# ── Evaluation Summary ───────────────────────────────────
import pandas as pd

eval_df = pd.DataFrame(results)
print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(eval_df[["test_id", "description", "passed", "keywords_found", "keywords_total"]].to_string(index=False))

accuracy = eval_df["passed"].mean() * 100
print(f"\nOverall Pass Rate: {accuracy:.0f}% ({eval_df['passed'].sum()}/{len(eval_df)})")

---
## 💬 Discussion Questions

1. **Why did the agent call `get_weather` even for "Paris" (not in our database)?** What does this tell you about tool selection?

2. **In the multi-tool test (T4), how many steps did the agent take?** Could it have been more efficient?

3. **How would you improve the test cases?** What scenarios are missing?

4. **What happens if you give vague tool descriptions?** Try changing a docstring and re-running.

---
## Wrap-up & Key Takeaways

**What you learned:**
- Function calling = model requests a call, your code executes it
- AUTO/ANY/NONE modes control tool usage aggressiveness
- The agent loop: history → generate → execute tools → repeat
- Multi-tool agents: the model picks the right tool(s) for each query
- Test cases with expected keywords and tools catch regressions

**What's next:** In the **Independent Lab**, you'll build your own domain-specific agent with 3-4 custom tools, test it, and improve it.

In [ ]:
# ── Export Prompt Log ────────────────────────────────────
import pandas as pd

if PROMPT_LOG:
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day4_lab1_prompt_log.csv", index=False)
    print(f"✅ Exported {len(log_df)} log entries to day4_lab1_prompt_log.csv")
else:
    print("No interactions logged.")

In [ ]:
# ── Export Agent Trace as JSONL ───────────────────────────
# JSONL (one JSON object per line) is ideal for debugging agent behavior:
# - Easy to grep for specific tool calls or errors
# - Can be loaded with pd.read_json(path, lines=True)
# - Each line is self-contained (no array structure to break)

def save_trace(trace_data, path):
    """Save a list of dicts as JSONL (one JSON object per line)."""
    with open(path, "w", encoding="utf-8") as f:
        for row in trace_data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"✅ Saved trace → {path}")

# Save all logged interactions as a JSONL trace
if PROMPT_LOG:
    save_trace(PROMPT_LOG, "day4_lab1_trace.jsonl")
else:
    print("No interactions to save.")

In [ ]:
# ── Session Statistics ───────────────────────────────────
tool_calls = sum(1 for log in PROMPT_LOG if log.get("label") == "tool_call")
agent_outputs = sum(1 for log in PROMPT_LOG if log.get("label") == "agent_output")

print(f"Session Statistics:")
print(f"  Total log entries:  {len(PROMPT_LOG)}")
print(f"  Tool calls:         {tool_calls}")
print(f"  Agent completions:  {agent_outputs}")
print(f"  Test cases run:     {len(test_cases)}")
print(f"  Pass rate:          {accuracy:.0f}%")